# 03 — Regresión Lineal Múltiple MCO (Parte 2)
**IA Generativa y Saber Pro 2021-2024** | Eduardo A. Victoria Cadena

Estima 6 modelos × 3 especificaciones. Errores HC3 robustos y clusterizados por IES.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUTA_PROYECTO = '/content/drive/MyDrive/IA-Y-EDUCACION-SUPERIOR'
import sys; sys.path.insert(0, RUTA_PROYECTO + '/python')
import subprocess, sys as _sys
for pkg in ['linearmodels']:
    subprocess.check_call([_sys.executable,'-m','pip','install',pkg,'-q'])

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
import warnings; warnings.filterwarnings('ignore')
from utils import (MODULOS_GENERICOS, ETIQUETAS_VARS, CONTROLES, DUMMIES_DEPTO,
                   ALPHA, guardar_tabla)

PROC_DIR = f'{RUTA_PROYECTO}/data/processed'
TBL_DIR  = f'{RUTA_PROYECTO}/outputs/tablas'
MDL_DIR  = f'{RUTA_PROYECTO}/outputs/modelos'
import os; os.makedirs(MDL_DIR, exist_ok=True)

df = pd.read_pickle(f'{PROC_DIR}/datos_analisis.pkl')
print(f"Datos cargados: {len(df):,} filas")

In [ ]:
# ── Construcción de la fórmula — análisis NACIONAL ───────────────────────
# Las dummies departamentales se generan automáticamente con C(depto_ies)
# con Bogotá D.C. como categoría de referencia (Treatment reference).
from utils import CONTROLES, DEPTO_REF

def construir_formula(var_dep, incluir_depto=True, incluir_dist=True, ef=None):
    """
    var_dep      : variable dependiente
    incluir_depto: True → C(depto_ies) con Bogotá como referencia
    incluir_dist : True → distancia_bogota_km como continua
    ef           : None | 'ies' (EF institución) | 'mun' (EF municipio)
    """
    rhs = ['periodo_ia']
    if incluir_depto and 'depto_ies' in df.columns:
        # statsmodels interpreta C(var, Treatment('ref')) como dummies con ref. explícita
        rhs.append(f"C(depto_ies, Treatment(reference='{DEPTO_REF}'))")
    if incluir_dist and 'distancia_bogota_km' in df.columns:
        rhs.append('distancia_bogota_km')
    rhs += [c for c in CONTROLES if c in df.columns]
    if ef == 'ies' and 'nombre_ies' in df.columns:
        rhs.append('C(nombre_ies)')
    elif ef == 'mun' and 'area_residencia' in df.columns:
        rhs.append('C(area_residencia)')
    return f'{var_dep} ~ ' + ' + '.join(rhs)

print("Fórmula de prueba:")
print(construir_formula('puntaje_saberpro_generico'))

In [ ]:
# ── Estimación con errores HC3 y clusterizados ────────────────────────────
from statsmodels.stats.sandwich_covariance import cov_cluster

def estimar_modelo(df, var_dep, formula, spec='base'):
    cols_need = [var_dep] + [c for c in CONTROLES + DUMMIES_DEPTO
                             + ['periodo_ia','distancia_bogota_km']
                             if c in df.columns]
    df_c = df[list(set(cols_need) & set(df.columns))].dropna()
    if len(df_c) < 30:
        print(f"  Muestra muy pequeña ({len(df_c)}) para {var_dep} ({spec})")
        return None

    mod = smf.ols(formula, data=df_c).fit()

    # HC3 robustos
    mod_hc3 = mod.get_robustcov_results(cov_type='HC3')

    # Clusterizados por IES (si disponible)
    if 'nombre_ies' in df_c.columns:
        groups = df_c['nombre_ies'].astype('category').cat.codes
        try:
            mod_cl = mod.get_robustcov_results(cov_type='cluster',
                                               groups=groups)
        except Exception:
            mod_cl = mod_hc3
    else:
        mod_cl = mod_hc3

    return {
        'var_dep' : var_dep,
        'spec'    : spec,
        'n'       : len(df_c),
        'modelo'  : mod,
        'mod_hc3' : mod_hc3,
        'mod_cl'  : mod_cl,
        'r2_adj'  : mod.rsquared_adj,
        'aic'     : mod.aic,
        'bic'     : mod.bic,
    }

In [ ]:
# ── Extraer coeficientes en tabla limpia ─────────────────────────────────
def extraer_coeficientes(res):
    if res is None: return None
    mod_cl = res['mod_cl']
    tbl = pd.DataFrame({
        'variable' : mod_cl.params.index,
        'coef'     : mod_cl.params.values,
        'se'       : mod_cl.bse.values,
        't_stat'   : mod_cl.tvalues.values,
        'p_valor'  : mod_cl.pvalues.values,
    })
    tbl['sig']     = tbl['p_valor'].apply(lambda p: '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '.' if p<0.10 else '')
    tbl['ic_95_l'] = tbl['coef'] - 1.96*tbl['se']
    tbl['ic_95_u'] = tbl['coef'] + 1.96*tbl['se']
    tbl['var_dep'] = res['var_dep']
    tbl['spec']    = res['spec']
    tbl['n']       = res['n']
    tbl['r2_adj']  = round(res['r2_adj'], 4)
    tbl[['coef','se','t_stat','p_valor','ic_95_l','ic_95_u']] = \
        tbl[['coef','se','t_stat','p_valor','ic_95_l','ic_95_u']].round(4)
    return tbl

In [ ]:
# ── Estimar 6 × 3 modelos ─────────────────────────────────────────────────
vars_dep = ['puntaje_saberpro_generico'] + [m for m in MODULOS_GENERICOS if m in df.columns]
specs    = {'base': None, 'ef_ies': 'ies', 'ef_mun': 'mun'}

todos_modelos = {}
for vd in vars_dep:
    for nombre_spec, ef_val in specs.items():
        key = f'{vd}__{nombre_spec}'
        print(f"  Estimando: {vd} | {nombre_spec}...")
        formula = construir_formula(vd, incluir_dummies=True,
                                    incluir_dist=True, ef=ef_val)
        res = estimar_modelo(df, vd, formula, spec=nombre_spec)
        todos_modelos[key] = res

print(f"\nModelos estimados: {len([v for v in todos_modelos.values() if v])}")

In [ ]:
# ── Tabla 4: coeficientes consolidados ───────────────────────────────────
vars_interes = ['periodo_ia','distancia_bogota_km'] + DUMMIES_DEPTO + CONTROLES + ['Intercept']

lista_tbls = []
for key, res in todos_modelos.items():
    tbl = extraer_coeficientes(res)
    if tbl is not None:
        tbl = tbl[tbl['variable'].isin(vars_interes) |
                  tbl['variable'].str.startswith('Intercept')]
        lista_tbls.append(tbl)

tabla4 = pd.concat(lista_tbls, ignore_index=True)
tabla4['etiqueta'] = tabla4['variable'].map(ETIQUETAS_VARS).fillna(tabla4['variable'])
guardar_tabla(tabla4, 'T4_coeficientes_todos_modelos', TBL_DIR)
print(f"Tabla 4 guardada: {len(tabla4)} filas")
tabla4.head(10)

In [ ]:
# ── Resumen β_IA por módulo y especificación ──────────────────────────────
beta_ia = (tabla4[tabla4['variable']=='periodo_ia']
           [['var_dep','spec','coef','se','t_stat','p_valor','sig','n','r2_adj']]
           .copy())
beta_ia['modulo']   = beta_ia['var_dep'].map(ETIQUETAS_VARS).fillna(beta_ia['var_dep'])
beta_ia['coef_ic']  = (beta_ia['coef'].round(3).astype(str) + ' ['
                      + (beta_ia['coef']-1.96*beta_ia['se']).round(3).astype(str) + ', '
                      + (beta_ia['coef']+1.96*beta_ia['se']).round(3).astype(str) + ']'
                      + beta_ia['sig'])
guardar_tabla(beta_ia, 'T4b_beta_IA_por_modulo', TBL_DIR)
print("\n=== β_IA por módulo y especificación ===")
print(beta_ia[['modulo','spec','coef_ic','p_valor','n','r2_adj']].to_string(index=False))

In [ ]:
# ── Tres versiones geográficas: solo dummies / solo dist / ambas ──────────
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_rows = []
for vd in vars_dep:
    for version, inc_d, inc_dist in [('solo_dummies',True,False),
                                      ('solo_distancia',False,True),
                                      ('ambas',True,True)]:
        formula = construir_formula(vd, incluir_dummies=inc_d,
                                    incluir_dist=inc_dist, ef=None)
        res = estimar_modelo(df, vd, formula, spec=version)
        if res is None: continue
        # VIF
        mod  = res['modelo']
        X    = sm.add_constant(mod.model.exog)
        cols = list(mod.model.exog_names)
        for i, c in enumerate(cols):
            try:
                v = variance_inflation_factor(X, i+1)
            except: v = np.nan
            vif_rows.append({'modulo':vd,'version':version,'variable':c,
                             'vif':round(v,2),'alerta':v>10})

tbl_vif = pd.DataFrame(vif_rows)
guardar_tabla(tbl_vif, 'T_vif_version_ambas', TBL_DIR)
print("\n=== VIF altos (> 10) ===")
print(tbl_vif[tbl_vif['alerta']].to_string(index=False) or "  Ninguno encontrado")